# Phase 02.00 — Multi-frame F1/F3/F8

This notebook evaluates the pinned zero-shot model and the Phase 01 LoRA r16 adapter with 1, 3, and 8 deterministically sampled frames. Every variant uses the exact frozen validation IDs. Each frame owns its own dynamic tiles, `num_patches_list` records those boundaries, and `image_flags` has one entry per tile.


In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)


Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


## 1. Configuration and frozen validation gate

Start with `DEBUG_LIMIT=20`. Set it to `None` for publishable runs. Debug and full artifacts are stored separately.


In [2]:
import gc
from peft import PeftModel

FRAME_COUNTS = [1, 3, 8]
DEBUG_LIMIT = 20
RUN_ZERO_SHOT = True
RUN_LORA_R16 = True

OUTPUT_ROOT = PROJECT_ROOT / "outputs/phase02/multiframe_f1_f3_f8"
VALIDATION_CSV = PATHS.phase1_split / "validation.csv"
phase1_status_path = PATHS.phase1_output / "PHASE01_STATUS.json"
assert phase1_status_path.is_file(), "Phase 01 status is missing"
phase1_status = json.loads(phase1_status_path.read_text(encoding="utf-8"))
UPSTREAM_RUN = phase1_status.get("run_name", "full")
ADAPTER_DIR = PATHS.lora / UPSTREAM_RUN / "checkpoints/adapter_final"

assert VALIDATION_CSV.is_file(), "Phase 01 frozen validation table is missing"
if RUN_LORA_R16:
    assert ADAPTER_DIR.is_dir(), f"Phase 01 LoRA adapter is missing: {ADAPTER_DIR}"
val_df = pd.read_csv(VALIDATION_CSV)
frozen_ids = json.loads((PATHS.phase1_split / "validation_sample_ids.json").read_text(encoding="utf-8"))
assert sorted(val_df.sample_id.astype(str)) == frozen_ids
run_scope = "debug20" if DEBUG_LIMIT else "full"
if RUN_LORA_R16:
    assert run_scope == UPSTREAM_RUN, f"Phase02 scope {run_scope} must match Phase01 adapter scope {UPSTREAM_RUN}"
print("Upstream Phase01 run:", UPSTREAM_RUN)
print("Rows per experiment:", min(len(val_df), DEBUG_LIMIT) if DEBUG_LIMIT else len(val_df))

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Upstream Phase01 run: debug20
Rows per experiment: 20


## 2. Reusable experiment runner

The runner checkpoints predictions and metrics after each frame count, so an interruption does not discard completed experiments.


In [3]:
def run_frame_grid(model, tokenizer, variant):
    results = []
    for frame_count in FRAME_COUNTS:
        print(f"\n[{variant}] F{frame_count}")
        predictions, metrics = evaluate_rows_multiframe(model, tokenizer, val_df, frame_count, limit=DEBUG_LIMIT)
        out_dir = OUTPUT_ROOT / variant / f"F{frame_count}" / run_scope
        out_dir.mkdir(parents=True, exist_ok=True)
        predictions.to_csv(out_dir / "predictions.csv", index=False)
        save_json(out_dir / "metrics.json", {**metrics, "variant": variant})
        save_json(out_dir / "config.json", {
            "variant": variant, "frame_count": frame_count, "model_id": MODEL_ID,
            "revision": MODEL_REVISION, "seed": SEED, "limit": DEBUG_LIMIT,
            "sampling": "equal_temporal_bin_midpoints", "max_dynamic_tiles_per_frame": MAX_DYNAMIC_TILES,
            "num_patches_list": "one entry per frame", "image_flags": "one per visual tile",
            "template": "Hermes-2", "flash_attention": False,
        })
        results.append({"variant": variant, **metrics})
        display(metrics)
    return results


## 3. Zero-shot F1/F3/F8


In [4]:
all_metrics = []
if RUN_ZERO_SHOT:
    zero_model, tokenizer = load_model_and_tokenizer(training=False)
    zero_model.language_model.config._attn_implementation = "sdpa"
    assert zero_model.language_model.config._attn_implementation == "sdpa"
    zero_model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    all_metrics.extend(run_frame_grid(zero_model, tokenizer, "zero_shot"))
    del zero_model
    gc.collect()
    torch.cuda.empty_cache()

/root/venvs/roadbuddy-rtx3090-py310/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.



[zero_shot] F1


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'frame_count': 8,
 'rows': 20,
 'accuracy': 0.6,
 'macro_f1': 0.6035353535353536,
 'parse_rate': 1.0,
 'mean_tiles': 52.8}

## 4. LoRA r16 F1/F3/F8


In [5]:
if RUN_LORA_R16:
    base_model, tokenizer = load_model_and_tokenizer(training=False)
    base_model.language_model.config._attn_implementation = "sdpa"
    assert base_model.language_model.config._attn_implementation == "sdpa"
    base_model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    lora_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    lora_model.eval()
    all_metrics.extend(run_frame_grid(lora_model, tokenizer, "lora_r16"))
    del lora_model, base_model
    gc.collect()
    torch.cuda.empty_cache()


[lora_r16] F1


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


{'frame_count': 8,
 'rows': 20,
 'accuracy': 0.6,
 'macro_f1': 0.6035353535353535,
 'parse_rate': 1.0,
 'mean_tiles': 52.8}

## 5. Comparison and phase status


In [6]:
summary = pd.DataFrame(all_metrics).sort_values(["variant", "frame_count"])
summary.to_csv(OUTPUT_ROOT / f"phase02_summary_{run_scope}.csv", index=False)
display(summary)

expected = {(variant, count) for variant in (["zero_shot"] if RUN_ZERO_SHOT else []) + (["lora_r16"] if RUN_LORA_R16 else []) for count in FRAME_COUNTS}
observed = set(zip(summary.variant, summary.frame_count))
assert observed == expected
expected_rows = min(len(val_df), DEBUG_LIMIT) if DEBUG_LIMIT else len(val_df)
assert summary.rows.eq(expected_rows).all()
best = summary.sort_values(["accuracy", "macro_f1"], ascending=False).iloc[0]
phase_status = "debug_complete" if DEBUG_LIMIT else "complete"
save_json(OUTPUT_ROOT / "PHASE02_STATUS.json", {
    "phase": "02", "status": phase_status, "run_name": run_scope,
    "upstream_run": UPSTREAM_RUN, "experiments": len(summary),
    "best_variant": best.variant, "best_frame_count": int(best.frame_count),
    "best_accuracy": float(best.accuracy), "validation_rows": expected_rows,
    "model_revision": MODEL_REVISION, "seed": SEED,
})
print(f"Phase 02 {run_scope}: {phase_status.upper()}")

,variant,frame_count,rows,accuracy,macro_f1,parse_rate,mean_tiles
3,lora_r16,1,20,0.60,0.616228,1.0,6.6
4,lora_r16,3,20,0.60,0.603535,1.0,19.8
5,lora_r16,8,20,0.60,0.603535,1.0,52.8
0,zero_shot,1,20,0.70,0.663690,1.0,6.6
1,zero_shot,3,20,0.65,0.596667,1.0,19.8
2,zero_shot,8,20,0.60,0.603535,1.0,52.8


Phase 02 debug20: DEBUG_COMPLETE
